In [ ]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"
if [ "$(uname -sm)" != "Linux x86_64" ]; then
    echo "EMProt requires x86-64 Linux." >&2
    exit 1
fi
mkdir -p "$work"

conda="$(command -v conda || printf '%s' "$work/miniforge/bin/conda")"
if [ ! -x "$conda" ]; then
    curl -fLo "$work/miniforge.sh" https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
    bash "$work/miniforge.sh" -b -p "$work/miniforge"
fi

repo="$work/EMProt"
if [ ! -d "$repo" ]; then
    git clone https://github.com/huang-laboratory/EMProt.git "$repo"
fi
git -C "$repo" checkout --quiet --detach 9489d52

env="$work/EMProt-env"
CONDA_CHANNEL_PRIORITY=flexible "$conda" env update -p "$env" -f "$repo/environment.yml"

cd "$repo"
"$conda" run -p "$env" python -m pip install -e .
if [ ! -d emprot/weights/model_all_atom ]; then
    curl -fL http://huanglab.phys.hust.edu.cn/EMProt/pretrained_weights/weights.tgz | tar -xz -C emprot
fi


In [ ]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"

conda="$(command -v conda || printf '%s' "$work/miniforge/bin/conda")"

method=EMProt
export MPLBACKEND=Agg
cd "$work/EMProt"
env="$work/EMProt-env"

for target in 22XP 32MC; do
    data="$root/data/targets/$target"
    map=("$data"/inputs/emd_*.map)
    fasta="$data/inputs/rcsb_pdb_${target}.fasta"
    out="$work/runs/$method/$target"
    mkdir -p "$out"

    cif=("$data"/predictions/af3/*/*_model_0.cif)
    "$conda" run -p "$env" --no-capture-output python - "${cif[0]}" "$out/af3_chain.pdb" <<'PYCODE'
import sys
from Bio.PDB import MMCIFParser, PDBIO

structure = MMCIFParser(QUIET=True).get_structure("af3", sys.argv[1])
writer = PDBIO()
writer.set_structure(next(structure.get_chains()))
writer.save(sys.argv[2])
PYCODE
    time "$conda" run -p "$env" --no-capture-output emprot build --map "${map[0]}" \
        --seq "$fasta" --chain "$out/af3_chain.pdb" --device 0 --output "$out"
done


In [ ]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"

if [ "$(uname -s)" = Linux ] && ! command -v g++ >/dev/null; then
    apt-get update
    apt-get install -y g++
fi

phenix_env="${PHENIX_ENV:-$work/phenix/phenix_env.sh}"
if [ ! -f "$phenix_env" ]; then
    for script in "$root/../cryo-benchmark-work/phenix/phenix_env.sh" "$HOME"/phenix-*/phenix_env.sh /Applications/phenix-*/phenix_env.sh; do
        if [ -f "$script" ]; then phenix_env="$script"; break; fi
    done
fi

method="EMProt"
if [ ! -d "$work/USalign" ]; then
    git clone https://github.com/pylelab/USalign.git "$work/USalign"
fi
make -C "$work/USalign" USalign

for target in 22XP 32MC; do
    out="$work/runs/$method/$target"
    reference="$root/data/targets/$target/reference/$target.cif"
    model="$out/output.cif"
    test -s "$model"
    "$work/USalign/USalign" "$model" "$reference" -mol prot -mm 1 -ter 1 -outfmt 2 > "$out/usalign.tsv"
    if [ -f "$phenix_env" ]; then
        (source "$phenix_env"; phenix.chain_comparison "$reference" "$model" \
            max_dist=3 unique_part_of_target_only=False test_unique_part_of_target_only=False \
            use_crystal_symmetry=False) > "$out/chain-comparison.txt"
    fi
done
